# 02 · Spectral analysis and group statistics

1. **Spectra** – per subject, channel and condition: multitaper (or Welch) PSD over the analysis
   window, averaged across epochs, `log10`-transformed (optionally 1/f-corrected), line-noise bins masked
2. **Band power** – average of the spectrum within each band in `spectral.bands`
3. **Group statistics** (subjects are the unit of observation; channels are averaged within subject):
   * **per frequency**: 1-D cluster-based permutation test (or paired t + FDR) on paired differences
   * **per band**: paired t-test / Wilcoxon, multiple-comparison correction across bands × contrasts × regions
   * **omnibus**: Friedman test across ≥3 conditions
   * **mixed model** (optional): channel-level `power ~ condition` with random intercepts for subject and channel
4. **Per location** (optional): all of the above repeated within each region of `regions_file`
   covered by at least `stats.per_region.min_subjects` subjects

Inputs are the epochs written by notebook 01. Outputs go to `derivatives/spectral/` and `derivatives/group/`.

## 0 · Parameters

In [ ]:
import os

CONFIG_PATH = os.environ.get("SEEG_CONFIG", "../config/config.yaml")
RECOMPUTE   = False      # True: recompute spectra even if cached in derivatives/spectral
SAVE_FIGS   = True

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
%matplotlib inline

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd

from seeg_wm import GroupStats, SpectralAnalyzer, load_config
from seeg_wm import plotting as splt
from seeg_wm.utils import get_logger, set_seed

cfg = load_config(CONFIG_PATH)
log = get_logger(cfg["project"]["log_level"])
set_seed(cfg["project"]["random_seed"])
mne.set_log_level("WARNING")
pd.set_option("display.width", 200)

CONDITIONS = list(cfg["spectral"]["conditions"])
CONTRASTS = cfg["stats"]["contrasts"]
print("conditions:", CONDITIONS)
print("contrasts :", [c["name"] for c in CONTRASTS])

def save(fig, name):
    if SAVE_FIGS:
        splt.savefig(fig, cfg, name)

## 1 · Per-subject spectra and band power

In [ ]:
analyzer = SpectralAnalyzer(cfg)
psd, band = analyzer.run(cfg.subjects, recompute=RECOMPUTE)

gdir = cfg.group_dir("group")
psd.to_csv(gdir / "group_psd.csv.gz", index=False, compression="gzip")
band.to_csv(gdir / "group_bandpower.csv", index=False)
print(f"{psd['subject'].nunique()} subjects, {psd.groupby('subject')['channel'].nunique().sum()} channels")
band.head()

In [ ]:
gs = GroupStats(cfg)
coverage = gs.coverage(band)
fig = splt.plot_coverage(coverage.reset_index())
save(fig, "coverage")

In [ ]:
# Single-subject sanity check
fig = splt.plot_subject_spectra(psd, cfg, cfg.subjects[0], conditions=CONDITIONS[:4])

## 2 · Grand-average spectra (mean ± SEM across subjects)

In [ ]:
fig = splt.plot_group_spectra(psd, cfg, CONDITIONS[:4])
save(fig, "group_spectra_all")

In [ ]:
regions = [r for r in coverage.index if r != cfg["stats"]["per_region"]["unassigned_label"]]
n = len(regions)
if n:
    ncol = min(3, n); nrow = int(np.ceil(n / ncol))
    fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3.3 * nrow), squeeze=False,
                             constrained_layout=True)
    for ax, r in zip(axes.flat, regions):
        splt.plot_group_spectra(psd, cfg, CONDITIONS[:4], region=r, ax=ax)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    save(fig, "group_spectra_regions")

## 3 · Group statistics

Runs every test in `stats` for the whole cohort (`region == "All"`) and, if `stats.per_region.enabled`,
for every sufficiently covered region. With *n* subjects, the smallest attainable permutation p-value is
1/2^(n-1).

In [ ]:
results = gs.run(psd, band)
paths = gs.save()
{k: v.shape for k, v in results.items()}

### 3a · Per frequency (whole cohort)

In [ ]:
curves, clusters = results.get("freq_curves", pd.DataFrame()), results["freq_clusters"]
if len(curves):
    fig, axes = plt.subplots(len(CONTRASTS), 1, figsize=(8, 3 * len(CONTRASTS)),
                             constrained_layout=True, squeeze=False)
    for ax, c in zip(axes[:, 0], CONTRASTS):
        sel = lambda d: d[(d["region"] == "All") & (d["contrast"] == c["name"])] if len(d) else d
        splt.plot_contrast_curve(sel(curves), sel(clusters), title=f"{c['a']} − {c['b']}",
                                 cfg=cfg, ax=ax)
    save(fig, "freq_contrasts_all")
display(clusters[clusters.get("significant", pd.Series(dtype=bool))].round(4)
        if len(clusters) else "no clusters")

### 3b · Per band (whole cohort)

In [ ]:
bt = results["band_tests"]
display(bt[bt["region"] == "All"][["contrast", "band", "n_subjects", "mean_a", "mean_b", "diff",
                                   "cohen_dz", "statistic", "p", "p_corrected", "significant"]].round(4))

In [ ]:
for c in CONTRASTS:
    t = bt[(bt["region"] == "All") & (bt["contrast"] == c["name"])]
    fig = splt.plot_band_paired(band, c["a"], c["b"], cfg=cfg, tests=t)
    save(fig, f"band_paired_{c['name']}_All")

### 3c · Omnibus (Friedman) and mixed models

In [ ]:
if "omnibus" in results and len(results["omnibus"]):
    display(results["omnibus"].round(4))

In [ ]:
if "mixed_models" in results:
    lm = results["mixed_models"]
    display(lm[lm["region"] == "All"].round(4))

## 4 · Per location

Effect size (Cohen's d_z) for each region × band. `*` marks significance after correction.

In [ ]:
if cfg["stats"]["per_region"]["enabled"]:
    for c in CONTRASTS:
        fig = splt.plot_region_band_heatmap(bt, c["name"], alpha=cfg["stats"]["alpha"],
                                            band_order=list(cfg["spectral"]["bands"]))
        save(fig, f"region_band_heatmap_{c['name']}")

In [ ]:
# Per-frequency contrasts per region
if cfg["stats"]["per_region"]["enabled"] and len(curves):
    regs = [r for r in curves["region"].unique() if r != "All"]
    for c in CONTRASTS:
        if not regs:
            break
        ncol = min(3, len(regs)); nrow = int(np.ceil(len(regs) / ncol))
        fig, axes = plt.subplots(nrow, ncol, figsize=(5 * ncol, 3 * nrow), squeeze=False,
                                 constrained_layout=True)
        for ax, r in zip(axes.flat, regs):
            cv = curves[(curves["region"] == r) & (curves["contrast"] == c["name"])]
            cl = clusters[(clusters["region"] == r) & (clusters["contrast"] == c["name"])] \
                if len(clusters) else None
            if len(cv):
                splt.plot_contrast_curve(cv, cl, title=r, cfg=cfg, ax=ax)
        for ax in axes.flat[len(regs):]:
            ax.set_visible(False)
        fig.suptitle(f"{c['a']} − {c['b']}")
        save(fig, f"freq_contrasts_regions_{c['name']}")

In [ ]:
# Significant region-level band effects
sig = bt[(bt["region"] != "All") & bt["significant"]]
display(sig[["region", "contrast", "band", "n_subjects", "diff", "cohen_dz", "p_corrected"]].round(4))

## 5 · Outputs

In [ ]:
print("statistics:", cfg.group_dir("group/stats"))
print("figures   :", cfg.group_dir("group/figures"))
sorted(p.name for p in cfg.group_dir("group").rglob("*.*"))